In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
train_df.head()

import wandb

In [3]:
train_df['answer'].value_counts() 

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

# MODEL 1

In [10]:
import os, re, random, collections, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

device: cuda


In [11]:
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"

OPTION_COLS = ["A", "B", "C", "D", "E"]
LABEL2IDX = {c: i for i, c in enumerate(OPTION_COLS)}
IDX2LABEL = {i: c for c, i in LABEL2IDX.items()}

def clean_scratch(t):
    return re.sub(r"\s+", " ", str(t)).strip()

train_df_scratch = pd.read_csv(f"{DATA_DIR}/train.csv")
train_df_scratch = train_df_scratch.dropna(subset=["prompt","A","B","C","D","E","answer"]).drop_duplicates()
train_df_scratch["prompt"] = train_df_scratch["prompt"].apply(clean_scratch)
for c in OPTION_COLS:
    train_df_scratch[c] = train_df_scratch[c].apply(clean_scratch)
train_df_scratch["label"] = train_df_scratch["answer"].map(LABEL2IDX).astype(int)

test_df_scratch = pd.read_csv(f"{DATA_DIR}/test.csv")
test_df_scratch["prompt"] = test_df_scratch["prompt"].apply(clean_scratch)
for c in OPTION_COLS:
    test_df_scratch[c] = test_df_scratch[c].apply(clean_scratch)

train_split_scratch, val_split_scratch = train_test_split(
    train_df_scratch, test_size=0.15, random_state=42, stratify=train_df_scratch["label"]
)
train_split_scratch = train_split_scratch.reset_index(drop=True)
val_split_scratch = val_split_scratch.reset_index(drop=True)
print("train:", len(train_split_scratch), "val:", len(val_split_scratch))


train: 1700 val: 300


In [12]:

def map_at_3_scratch(logits, labels):
    order = np.argsort(-logits, axis=1)[:, :3]
    scores = np.zeros(len(labels))
    for i, lab in enumerate(labels):
        hits = np.where(order[i] == lab)[0]
        if len(hits):
            scores[i] = 1.0 / (hits[0] + 1)
    return scores.mean()

def evaluate_all_scratch(logits, labels):
    preds = logits.argmax(1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro"),
            "map@3": map_at_3_scratch(logits, labels)}

def top3_labels_scratch(logits):
    order = np.argsort(-logits, axis=1)[:, :3]
    return [" ".join(IDX2LABEL[i] for i in row) for row in order]

In [13]:
TOKEN_RE = re.compile(r"[A-Za-z]+|\d+|[^\sA-Za-z\d]")
def tokenize_scratch(t):
    return TOKEN_RE.findall(t.lower())

class VocabScratch:
    def __init__(self, min_freq=2, max_size=30000):
        self.min_freq = min_freq
        self.max_size = max_size
        self.stoi = {"<pad>": 0, "<unk>": 1}
    def build(self, texts):
        counter = collections.Counter()
        for t in texts:
            counter.update(tokenize_scratch(t))
        for tok, freq in counter.most_common():
            if freq < self.min_freq or len(self.stoi) >= self.max_size:
                continue
            if tok not in self.stoi:
                self.stoi[tok] = len(self.stoi)
        return self
    def encode(self, text, max_len):
        ids = [self.stoi.get(t, 1) for t in tokenize_scratch(text)][:max_len]
        return ids + [0] * (max_len - len(ids))
    def __len__(self):
        return len(self.stoi)

texts_scratch = list(train_split_scratch["prompt"])
for c in OPTION_COLS:
    texts_scratch += list(train_split_scratch[c])
vocab_scratch = VocabScratch().build(texts_scratch)
print("vocab size:", len(vocab_scratch))
json.dump(vocab_scratch.stoi, open("/kaggle/working/vocab_scratch.json", "w"))

class MCQDatasetScratch(Dataset):
    def __init__(self, df, vocab, qlen=64, olen=96):
        self.df, self.vocab, self.qlen, self.olen = df, vocab, qlen, olen
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        q = torch.tensor(self.vocab.encode(row["prompt"], self.qlen))
        opts = [torch.tensor(self.vocab.encode(row[c], self.olen)) for c in OPTION_COLS]
        return q, opts, torch.tensor(row["label"])

def collate_scratch(batch):
    q = torch.stack([b[0] for b in batch])
    opts = [torch.stack([b[1][i] for b in batch]) for i in range(5)]
    labels = torch.stack([b[2] for b in batch])
    return q, opts, labels

train_loader_scratch = DataLoader(MCQDatasetScratch(train_split_scratch, vocab_scratch), batch_size=32, shuffle=True, collate_fn=collate_scratch)
val_loader_scratch = DataLoader(MCQDatasetScratch(val_split_scratch, vocab_scratch), batch_size=32, collate_fn=collate_scratch)

vocab size: 3001


In [14]:
class Model1ConvAttention(nn.Module):
    def __init__(self, vocab_size, emb_dim=150, conv_channels=150, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.conv = nn.Conv1d(emb_dim, conv_channels, kernel_size=3, padding=1)
        self.conv_act = nn.ReLU()
        self.attn_proj = nn.Linear(conv_channels, 1)
        pooled_dim = conv_channels * 3
        combo_dim = pooled_dim * 4
        self.scorer = nn.Sequential(
            nn.Linear(combo_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.LayerNorm(hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
    def encode(self, ids):
        mask = (ids != 0).unsqueeze(-1).float()
        emb = self.embedding(ids)
        conv_out = self.conv_act(self.conv(emb.transpose(1, 2))).transpose(1, 2)
        feats = conv_out * mask
        mean_pool = feats.sum(1) / mask.sum(1).clamp(min=1e-6)
        max_pool = feats.masked_fill(mask == 0, -1e9).max(1).values
        attn_scores = self.attn_proj(feats).squeeze(-1).masked_fill(mask.squeeze(-1) == 0, -1e9)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        attn_pool = (feats * attn_weights).sum(1)
        return torch.cat([mean_pool, max_pool, attn_pool], dim=-1)
    def score(self, q_vec, o_vec):
        combo = torch.cat([q_vec, o_vec, q_vec * o_vec, torch.abs(q_vec - o_vec)], dim=-1)
        return self.scorer(combo).squeeze(-1)
    def forward(self, q_ids, opt_ids_list):
        q_vec = self.encode(q_ids)
        return torch.stack([self.score(q_vec, self.encode(o)) for o in opt_ids_list], dim=1)

model1_scratch = Model1ConvAttention(len(vocab_scratch)).to(device)

In [15]:
import wandb
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret("wandb"))

wandb.init(project="smart-mcq-solver", name="model1_conv_attention_final", reinit=True)
opt_model1 = torch.optim.Adam(model1_scratch.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn_scratch = nn.CrossEntropyLoss()

best_map3_model1 = -1
for epoch in range(40):
    model1_scratch.train()
    total_loss = 0
    for q, opts, labels in train_loader_scratch:
        q, opts, labels = q.to(device), [o.to(device) for o in opts], labels.to(device)
        logits = model1_scratch(q, opts)
        loss = loss_fn_scratch(logits, labels)
        opt_model1.zero_grad(); loss.backward(); opt_model1.step()
        total_loss += loss.item() * len(labels)

    model1_scratch.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for q, opts, labels in val_loader_scratch:
            q, opts = q.to(device), [o.to(device) for o in opts]
            logits = model1_scratch(q, opts)
            all_logits.append(logits.cpu().numpy()); all_labels.append(labels.numpy())
    logits = np.concatenate(all_logits); labels_np = np.concatenate(all_labels)
    m = evaluate_all_scratch(logits, labels_np)
    print(epoch, "train_loss:", total_loss/len(train_split_scratch), m)
    wandb.log({"epoch": epoch, "train_loss": total_loss/len(train_split_scratch), **{f"val_{k}": v for k, v in m.items()}})
    if m["map@3"] > best_map3_model1:
        best_map3_model1 = m["map@3"]
        torch.save(model1_scratch.state_dict(), "/kaggle/working/model1_scratch_best.pt")

# wandb.summary["best_val_map@3"] = best_map3_model1
# wandb.finish()
# print("Model 1 best val MAP@3:", best_map3_model1)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aribasayed2005 (24f3004086-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


0 train_loss: 1.6065416319230024 {'accuracy': 0.8033333333333333, 'macro_f1': 0.795719106098421, 'map@3': np.float64(0.866111111111111)}
1 train_loss: 0.6356578996887102 {'accuracy': 0.98, 'macro_f1': 0.9798758552551252, 'map@3': np.float64(0.9916666666666667)}
2 train_loss: 0.10846580671036944 {'accuracy': 0.9933333333333333, 'macro_f1': 0.9932695923128417, 'map@3': np.float64(0.9983333333333333)}
3 train_loss: 0.04999018924201236 {'accuracy': 0.9966666666666667, 'macro_f1': 0.9966919922006319, 'map@3': np.float64(1.0)}
4 train_loss: 0.027179283828967633 {'accuracy': 0.9966666666666667, 'macro_f1': 0.9966919922006319, 'map@3': np.float64(1.0)}
5 train_loss: 0.016350104142637815 {'accuracy': 0.9966666666666667, 'macro_f1': 0.9966919922006319, 'map@3': np.float64(0.9983333333333333)}
6 train_loss: 0.014213167752194054 {'accuracy': 0.9933333333333333, 'macro_f1': 0.9934781370306055, 'map@3': np.float64(0.9983333333333333)}
7 train_loss: 0.009196910626008449 {'accuracy': 0.996666666666666

In [16]:
def predict_scratch_model(model, qlen=64, olen=96, batch_size=32):
    model.eval()
    all_logits = []
    with torch.no_grad():
        for start in range(0, len(test_df_scratch), batch_size):
            batch = test_df_scratch.iloc[start:start+batch_size]
            q_ids = torch.tensor([vocab_scratch.encode(t, qlen) for t in batch["prompt"]]).to(device)
            opt_ids = [torch.tensor([vocab_scratch.encode(t, olen) for t in batch[c]]).to(device) for c in OPTION_COLS]
            logits = model(q_ids, opt_ids)
            all_logits.append(logits.cpu().numpy())
    return np.concatenate(all_logits)

model1_scratch.load_state_dict(torch.load("/kaggle/working/model1_scratch_best.pt", map_location=device))
test_logits_model1 = predict_scratch_model(model1_scratch)
submission = pd.DataFrame()
submission["ID"] = test_df_scratch["id"]
submission["Prediction"] = top3_labels_scratch(test_logits_model1)
submission.to_csv("/kaggle/working/submission.csv", index=False)


# MODEL 2


In [17]:
import os, re, random, collections, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

device: cuda


In [18]:
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"  # adjust if needed

OPTION_COLS = ["A", "B", "C", "D", "E"]
LABEL2IDX = {c: i for i, c in enumerate(OPTION_COLS)}
IDX2LABEL = {i: c for c, i in LABEL2IDX.items()}

def clean_text_roberta(t):
    return re.sub(r"\s+", " ", str(t)).strip()

train_df_roberta = pd.read_csv(f"{DATA_DIR}/train.csv")
train_df_roberta = train_df_roberta.dropna(subset=["prompt", "A", "B", "C", "D", "E", "answer"]).drop_duplicates()
train_df_roberta["prompt"] = train_df_roberta["prompt"].apply(clean_text_roberta)
for c in OPTION_COLS:
    train_df_roberta[c] = train_df_roberta[c].apply(clean_text_roberta)
train_df_roberta["label"] = train_df_roberta["answer"].map(LABEL2IDX)
assert train_df_roberta["label"].isna().sum() == 0
train_df_roberta["label"] = train_df_roberta["label"].astype(int)

test_df_roberta = pd.read_csv(f"{DATA_DIR}/test.csv")
test_df_roberta["prompt"] = test_df_roberta["prompt"].apply(clean_text_roberta)
for c in OPTION_COLS:
    test_df_roberta[c] = test_df_roberta[c].apply(clean_text_roberta)

print("train rows:", len(train_df_roberta), "test rows:", len(test_df_roberta))

train rows: 2000 test rows: 500


In [19]:
def map_at_3_roberta(logits, labels):
    order = np.argsort(-logits, axis=1)[:, :3]
    scores = np.zeros(len(labels))
    for i, lab in enumerate(labels):
        hits = np.where(order[i] == lab)[0]
        if len(hits):
            scores[i] = 1.0 / (hits[0] + 1)
    return scores.mean()

def evaluate_all_roberta(logits, labels):
    preds = logits.argmax(1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "map@3": map_at_3_roberta(logits, labels),
    }

def top3_labels_roberta(logits):
    order = np.argsort(-logits, axis=1)[:, :3]
    return [" ".join(IDX2LABEL[i] for i in row) for row in order]

def shuffled_option_val_roberta(df, seed=123):
    df2 = df.copy()
    for i in df2.index:
        cols = OPTION_COLS.copy()
        random.Random(seed + i).shuffle(cols)
        orig_vals = {c: df2.loc[i, c] for c in OPTION_COLS}
        new_label = None
        for new_pos, old_col in enumerate(cols):
            df2.loc[i, OPTION_COLS[new_pos]] = orig_vals[old_col]
            if old_col == OPTION_COLS[df.loc[i, "label"]]:
                new_label = new_pos
        df2.loc[i, "label"] = new_label
    return df2

In [20]:
class MCQDatasetBertRoberta(Dataset):
    def __init__(self, df, tok, max_len=192):
        self.df, self.tok, self.max_len = df, tok, max_len
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        q = [row["prompt"]] * 5
        opts = [row[c] for c in OPTION_COLS]
        enc = self.tok(
            q, opts, truncation="only_second",
            max_length=self.max_len, padding="max_length", return_tensors="pt"
        )
        enc = dict(enc)
        enc["labels"] = torch.tensor(row["label"])
        return enc

def bert_collate_roberta(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}

In [21]:
import wandb
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret("wandb"))

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [22]:
def train_roberta_fold(fold_train, fold_val, fold_id, epochs=3, lr=1e-5, batch_size=8,
                        max_len=192, grad_clip=1.0, warmup_ratio=0.1, weight_decay=0.01):
    tokenizer_roberta = AutoTokenizer.from_pretrained("roberta-base")
    model_roberta = AutoModelForMultipleChoice.from_pretrained("roberta-base").to(device)

    train_loader_roberta = DataLoader(
        MCQDatasetBertRoberta(fold_train, tokenizer_roberta, max_len),
        batch_size=batch_size, shuffle=True, collate_fn=bert_collate_roberta
    )
    val_loader_roberta = DataLoader(
        MCQDatasetBertRoberta(fold_val, tokenizer_roberta, max_len),
        batch_size=batch_size, collate_fn=bert_collate_roberta
    )

    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_params = [
        {"params": [p for n, p in model_roberta.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": weight_decay},
        {"params": [p for n, p in model_roberta.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ]
    optimizer_roberta = torch.optim.AdamW(optimizer_params, lr=lr)

    total_steps = len(train_loader_roberta) * epochs
    scheduler_roberta = get_linear_schedule_with_warmup(optimizer_roberta, int(total_steps * warmup_ratio), total_steps)

    wandb.init(project="smart-mcq-solver", name=f"model2_roberta_fold{fold_id}", reinit=True)
    best_map3_fold = -1
    best_state_fold = None

    for epoch in range(epochs):
        model_roberta.train()
        total_loss, n_seen = 0, 0
        for batch in train_loader_roberta:
            labels = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model_roberta(**batch, labels=labels)
            loss = out.loss
            if torch.isnan(loss):
                optimizer_roberta.zero_grad(); continue
            optimizer_roberta.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model_roberta.parameters(), grad_clip)
            optimizer_roberta.step(); scheduler_roberta.step()
            total_loss += loss.item() * len(labels); n_seen += len(labels)

        model_roberta.eval()
        all_logits, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader_roberta:
                labels = batch.pop("labels")
                batch = {k: v.to(device) for k, v in batch.items()}
                out = model_roberta(**batch)
                all_logits.append(out.logits.cpu().numpy()); all_labels.append(labels.numpy())
        logits = np.concatenate(all_logits); labels_np = np.concatenate(all_labels)
        m = evaluate_all_roberta(logits, labels_np)
        avg_loss = total_loss / max(n_seen, 1)
        print(f"fold {fold_id} epoch {epoch}: train_loss={avg_loss:.4f}", m)
        wandb.log({"epoch": epoch, "train_loss": avg_loss, **{f"val_{k}": v for k, v in m.items()}})

        if m["map@3"] > best_map3_fold:
            best_map3_fold = m["map@3"]
            best_state_fold = {k: v.cpu().clone() for k, v in model_roberta.state_dict().items()}

    model_roberta.load_state_dict(best_state_fold)
    wandb.summary["best_val_map@3"] = best_map3_fold
    wandb.finish()
    return model_roberta, tokenizer_roberta, best_map3_fold

In [23]:
def predict_roberta_test(model_roberta, tokenizer_roberta, test_df, max_len=192, batch_size=16):
    model_roberta.eval()
    all_logits = []
    with torch.no_grad():
        for start in range(0, len(test_df), batch_size):
            batch = test_df.iloc[start:start+batch_size]
            first = sum([[q]*5 for q in batch["prompt"]], [])
            second = sum([[batch.iloc[j][c] for c in OPTION_COLS] for j in range(len(batch))], [])
            enc = tokenizer_roberta(first, second, truncation="only_second", max_length=max_len,
                                     padding="max_length", return_tensors="pt")
            enc = {k: v.view(len(batch), 5, -1).to(device) for k, v in enc.items()}
            out = model_roberta(**enc)
            all_logits.append(out.logits.cpu().numpy())
    return np.concatenate(all_logits)

In [24]:
skf_roberta = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_test_logits_roberta = []
fold_val_scores_roberta = []
fold_models_roberta = []

for fold_id, (train_idx, val_idx) in enumerate(skf_roberta.split(train_df_roberta, train_df_roberta["label"])):
    fold_train_roberta = train_df_roberta.iloc[train_idx].reset_index(drop=True)
    fold_val_roberta = train_df_roberta.iloc[val_idx].reset_index(drop=True)

    model_roberta_fold, tokenizer_roberta_fold, val_map3_fold = train_roberta_fold(
        fold_train_roberta, fold_val_roberta, fold_id
    )
    fold_val_scores_roberta.append(val_map3_fold)
    fold_models_roberta.append((model_roberta_fold, tokenizer_roberta_fold))

    test_logits_fold = predict_roberta_test(model_roberta_fold, tokenizer_roberta_fold, test_df_roberta)
    fold_test_logits_roberta.append(test_logits_fold)

    torch.cuda.empty_cache()

print("fold val MAP@3 scores:", fold_val_scores_roberta)
print("mean:", np.mean(fold_val_scores_roberta), "std:", np.std(fold_val_scores_roberta))

test_logits_roberta_ensemble = np.mean(fold_test_logits_roberta, axis=0)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.weight               | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 
roberta.pooler.dense.weight     | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold 4 epoch 0: train_loss=1.5804 {'accuracy': 0.5425, 'macro_f1': 0.5392052405355876, 'map@3': np.float64(0.67875)}
fold 4 epoch 1: train_loss=1.2660 {'accuracy': 0.7875, 'macro_f1': 0.7849584606242839, 'map@3': np.float64(0.8520833333333334)}
fold 4 epoch 2: train_loss=0.8784 {'accuracy': 0.835, 'macro_f1': 0.8329798526981627, 'map@3': np.float64(0.88625)}


epoch,▁▅█
train_loss,█▅▁
val_accuracy,▁▇█
val_macro_f1,▁▇█
val_map@3,▁▇█
best_val_map@3,0.88625
epoch,2
train_loss,0.87839
val_accuracy,0.835
val_macro_f1,0.83298
val_map@3,0.88625


fold val MAP@3 scores: [np.float64(0.8920833333333333), np.float64(0.8804166666666666), np.float64(0.8491666666666667), np.float64(0.9520833333333333), np.float64(0.88625)]
mean: 0.892 std: 0.03350207290435753


In [25]:
model_roberta_check, tokenizer_roberta_check = fold_models_roberta[0]
val0_roberta = train_df_roberta.iloc[
    list(skf_roberta.split(train_df_roberta, train_df_roberta["label"]))[0][1]
].reset_index(drop=True)

normal_logits_roberta = predict_roberta_test(model_roberta_check, tokenizer_roberta_check, val0_roberta)
normal_m_roberta = evaluate_all_roberta(normal_logits_roberta, val0_roberta["label"].values)

shuffled_val_roberta = shuffled_option_val_roberta(val0_roberta)
shuf_logits_roberta = predict_roberta_test(model_roberta_check, tokenizer_roberta_check, shuffled_val_roberta)
shuf_m_roberta = evaluate_all_roberta(shuf_logits_roberta, shuffled_val_roberta["label"].values)

print("normal:", normal_m_roberta)
print("shuffled:", shuf_m_roberta)

normal: {'accuracy': 0.8375, 'macro_f1': 0.840132221206557, 'map@3': np.float64(0.8920833333333333)}
shuffled: {'accuracy': 0.8375, 'macro_f1': 0.8365155337357851, 'map@3': np.float64(0.8920833333333333)}


In [26]:
test_predictions_roberta = top3_labels_roberta(test_logits_roberta_ensemble)

print("Fingerprint check -- first 10 RoBERTa predictions:", test_predictions_roberta[:10])

submission = pd.DataFrame()
submission["ID"] = test_df_roberta["id"]
submission["Prediction"] = test_predictions_roberta
submission.to_csv("/kaggle/working/submission.csv", index=False)
submission.head()

Fingerprint check -- first 10 RoBERTa predictions: ['A D B', 'B A D', 'B E C', 'E C A', 'C A D', 'D C B', 'E C D', 'B E A', 'C E D', 'B E C']


,ID,Prediction
0,1,A D B
1,2,B A D
2,3,B E C
3,4,E C A
4,5,C A D


In [27]:
!pip install -q huggingface_hub

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("hf_token")
login(token=hf_token)
print("Logged in successfully")

Logged in successfully


In [28]:
import os
print(os.listdir("/kaggle/working"))

['.virtual_documents', 'model1_scratch_best.pt', 'wandb', 'submission.csv', 'vocab_scratch.json']


In [108]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# reuse your existing preprocessing -- adjust DATA_DIR if needed
train_df = train_df_scratch if "train_df_scratch" in dir() else None
assert train_df is not None, "run your data-loading cell first"

train_split, val_split = train_test_split(
    train_df, test_size=0.15, random_state=42, stratify=train_df["label"]
)
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model = AutoModelForMultipleChoice.from_pretrained("roberta-base").to(device)

train_loader = DataLoader(MCQDatasetBertRoberta(train_split, tokenizer, 128), batch_size=8, shuffle=True, collate_fn=bert_collate_roberta)
val_loader = DataLoader(MCQDatasetBertRoberta(val_split, tokenizer, 128), batch_size=8, collate_fn=bert_collate_roberta)

no_decay = ["bias", "LayerNorm.weight"]
optimizer_params = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": 0.01},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
]
opt = torch.optim.AdamW(optimizer_params, lr=1e-5)
total_steps = len(train_loader) * 2
scheduler = get_linear_schedule_with_warmup(opt, int(total_steps * 0.1), total_steps)

best_map3, best_state = -1, None
for epoch in range(2):
    model.train()
    for batch in train_loader:
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch, labels=labels)
        loss = out.loss
        if torch.isnan(loss):
            opt.zero_grad(); continue
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); scheduler.step()

    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            labels = batch.pop("labels")
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            all_logits.append(out.logits.cpu().numpy()); all_labels.append(labels.numpy())
    logits = np.concatenate(all_logits); labels_np = np.concatenate(all_labels)
    m = evaluate_all(logits, labels_np)
    print("epoch", epoch, m)
    if m["map@3"] > best_map3:
        best_map3 = m["map@3"]
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print("Best val MAP@3:", best_map3)

# save locally right away
save_dir = "/kaggle/working/model2_roberta_deploy"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved to", save_dir)

cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.weight               | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 
roberta.pooler.dense.weight     | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


NameError: name 'evaluate_all' is not defined

# MODEL 3

In [29]:
# class Model3TextCNN(nn.Module):
#     def __init__(self, vocab_size, emb_dim=128, num_filters=100, kernel_sizes=(2,3,4,5), dropout=0.3):
#         super().__init__()
#         self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
#         self.convs = nn.ModuleList([nn.Conv1d(emb_dim, num_filters, k, padding=k // 2) for k in kernel_sizes])
#         self.dropout = nn.Dropout(dropout)
#         out_dim = num_filters * len(kernel_sizes)
#         combo_dim = out_dim * 4
#         self.scorer = nn.Sequential(nn.Linear(combo_dim, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
#     def encode(self, ids):
#         emb = self.dropout(self.embedding(ids)).transpose(1, 2)
#         pooled = [torch.max(torch.relu(conv(emb)), dim=2).values for conv in self.convs]
#         return torch.cat(pooled, dim=-1)
#     def score(self, q_vec, o_vec):
#         combo = torch.cat([q_vec, o_vec, q_vec * o_vec, torch.abs(q_vec - o_vec)], dim=-1)
#         return self.scorer(combo).squeeze(-1)
#     def forward(self, q_ids, opt_ids_list):
#         q_vec = self.encode(q_ids)
#         return torch.stack([self.score(q_vec, self.encode(o)) for o in opt_ids_list], dim=1)

# model3_scratch = Model3TextCNN(len(vocab_scratch)).to(device)

In [30]:
# wandb.init(project="smart-mcq-solver", name="model3_textcnn_final", reinit=True)
# opt_model3 = torch.optim.Adam(model3_scratch.parameters(), lr=1e-3, weight_decay=1e-5)

# best_map3_model3 = -1
# for epoch in range(100):
#     model3_scratch.train()
#     total_loss = 0
#     for q, opts, labels in train_loader_scratch:
#         q, opts, labels = q.to(device), [o.to(device) for o in opts], labels.to(device)
#         logits = model3_scratch(q, opts)
#         loss = loss_fn_scratch(logits, labels)
#         opt_model3.zero_grad(); loss.backward(); opt_model3.step()
#         total_loss += loss.item() * len(labels)

#     model3_scratch.eval()
#     all_logits, all_labels = [], []
#     with torch.no_grad():
#         for q, opts, labels in val_loader_scratch:
#             q, opts = q.to(device), [o.to(device) for o in opts]
#             logits = model3_scratch(q, opts)
#             all_logits.append(logits.cpu().numpy()); all_labels.append(labels.numpy())
#     logits = np.concatenate(all_logits); labels_np = np.concatenate(all_labels)
#     m = evaluate_all_scratch(logits, labels_np)
#     print(epoch, "train_loss:", total_loss/len(train_split_scratch), m)
#     wandb.log({"epoch": epoch, "train_loss": total_loss/len(train_split_scratch), **{f"val_{k}": v for k, v in m.items()}})
#     if m["map@3"] > best_map3_model3:
#         best_map3_model3 = m["map@3"]
#         torch.save(model3_scratch.state_dict(), "/kaggle/working/model3_scratch_best.pt")

# wandb.summary["best_val_map@3"] = best_map3_model3
# wandb.finish()
# print("Model 3 best val MAP@3:", best_map3_model3)

In [31]:
# def predict_scratch_model(model, qlen=64, olen=96, batch_size=32):
#     model.eval()
#     all_logits = []
#     with torch.no_grad():
#         for start in range(0, len(test_df_scratch), batch_size):
#             batch = test_df_scratch.iloc[start:start+batch_size]
#             q_ids = torch.tensor([vocab_scratch.encode(t, qlen) for t in batch["prompt"]]).to(device)
#             opt_ids = [torch.tensor([vocab_scratch.encode(t, olen) for t in batch[c]]).to(device) for c in OPTION_COLS]
#             logits = model(q_ids, opt_ids)
#             all_logits.append(logits.cpu().numpy())
#     return np.concatenate(all_logits)


# model3_scratch.load_state_dict(torch.load("/kaggle/working/model3_scratch_best.pt", map_location=device))
# test_logits_model3 = predict_scratch_model(model3_scratch)
# submission = pd.DataFrame()
# submission["ID"] = test_df_scratch["id"]
# submission["Prediction"] = top3_labels_scratch(test_logits_model3)
# submission.to_csv("/kaggle/working/submission.csv", index=False)

# submission.head()

# BASELINE MODEL

In [32]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity

In [33]:
# train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [34]:
# train.describe()

In [35]:
# train["answer"].value_counts()

In [36]:
# train.info()


Dataset Observations

- Number of samples in training set: 2000
- Number of columns: 8 
- Missing values: null
- Question types: science related
- Answer distribution: balanced

In [37]:
# train.shape

In [38]:
# train.head(10)

In [39]:
# question = train.loc[0, "prompt"]
# question

In [40]:
# def clean_text(text):
#     text = str(text)          # Ensure it's a string
#     text = text.lower()       # Lowercase
#     text = text.strip()       # Remove extra spaces
#     return text

In [41]:
# train["prompt"] = train["prompt"].apply(clean_text)
# test["prompt"] = test["prompt"].apply(clean_text)

In [42]:
# option_columns = ["A", "B", "C", "D", "E"]
# for col in option_columns:
#     train[col] = train[col].apply(clean_text)
#     test[col] = test[col].apply(clean_text)

In [43]:
# train.head()

In [44]:
# train["question_length"] = train["prompt"].apply(len)
# train["question_length"].describe()

In [45]:
# corpus = []

# corpus.extend(train["prompt"].tolist())

# for col in ["A","B","C","D","E"]:
#     corpus.extend(train[col].tolist())

In [46]:
# vectorizer = TfidfVectorizer(
#     stop_words="english",
#     max_features=30000
# )
# vectorizer.fit(corpus)

In [47]:
# question_vectors = vectorizer.transform(train["prompt"])
# question_vectors 

In [48]:
# option_vectors = {}

# for option in ["A","B","C","D","E"]:
#     option_vectors[option] = vectorizer.transform(train[option])

In [49]:
# predictions = []

# for i in range(len(train)):

#     question = question_vectors[i]

#     scores = {}

#     for option in ["A","B","C","D","E"]:

#         scores[option] = cosine_similarity(
#             question,
#             option_vectors[option][i]
#         )[0][0]

#     sorted_scores = sorted(
#         scores.items(),
#         key=lambda x:x[1],
    #     reverse=True
    # )

    # top3 = " ".join(
    #     [x[0] for x in sorted_scores[:3]]
    # )

    # predictions.append(top3)

In [50]:
# test_predictions = []

# for i in range(len(test)):

#     question = question_vectors[i]

#     scores = {}

#     for option in ["A","B","C","D","E"]:

#         scores[option] = cosine_similarity(
#             question,
#             option_vectors[option][i]
#         )[0][0]

#     sorted_scores = sorted(
#         scores.items(),
#         key=lambda x:x[1],
#         reverse=True
#     )

#     top3 = " ".join(
#         [x[0] for x in sorted_scores[:3]]
#     )

    # test_predictions.append(top3)

In [51]:
#submission = pd.DataFrame()

#submission["id"] = test["id"]

#submission["Prediction"] = test_predictions

#submission.head()

# MILESTONES

* #  Milestone 1

In [52]:
# import pandas as pd

# sample = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
# print(sample.head())
# print(sample.columns)

In [53]:
# import pandas as pd
# import random

# sample = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

# choices = [
#     'A B C',
#     'A C B',
#     'B A C',
#     'B C A',
#     'C A B',
#     'C B A'
# ]

# sample['Prediction'] = [random.choice(choices) for _ in range(len(sample))]
# sample.to_csv('submission.csv', index=False)

In [54]:
# import pandas as pd
# import numpy as np
# import string
# from collections import Counter
# from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
# from sklearn.metrics.pairwise import cosine_similarity


Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  
*


In [55]:
# train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# freq = train["answer"].value_counts()

# q1 = freq.max() + freq.min()
# q1

After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv? 

In [56]:
# translator = str.maketrans('', '', string.punctuation)
# #The str.maketrans() method in Python creates a translation table (a dictionary-like mapping)
# #that specifies how characters in a string should be replaced, mapped, or deleted. 
# def clean_prompt(text):
#     text = str(text).lower()
#     text = text.translate(translator)
#     return text

# vocab = set()

# for text in train["prompt"]:
#     words = clean_prompt(text).split()
#     vocab.update(words)

# q2 = len(vocab)
# q2

Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [57]:
# row1 = train.iloc[0]
# words = clean_prompt(row1["prompt"]).split()
# filtered = [w for w in words if w not in ENGLISH_STOP_WORDS]

# q3 = len(filtered)
# q3


Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [58]:
# combined_docs = []

# for _, row in train.iterrows():
#     combined = " ".join([str(row["prompt"]),str(row["A"]),str(row["B"]),str(row["C"]),
#         str(row["D"]),str(row["E"])])
#     combined_docs.append(combined)
# vectorizer = TfidfVectorizer(stop_words="english")
# vectorizer.fit(combined_docs)

# q4 = len(vectorizer.get_feature_names_out())
# q4

Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [59]:
# prompt_vec = vectorizer.transform([row1["prompt"]])
# a_vec = vectorizer.transform([row1["A"]])

# sim = cosine_similarity(prompt_vec, a_vec)[0][0]

# q5 = round(sim, 4)
# q5


Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   
*


In [60]:
# correct = 0
# for _, row in train.iterrows():

#     p = vectorizer.transform([row["prompt"]])

#     scores = {}

#     for opt in ["A","B","C","D","E"]:
#         o = vectorizer.transform([row[opt]])
#         scores[opt] = cosine_similarity(p, o)[0][0]

#     pred = max(scores, key=scores.get)

#     if pred == row["answer"]:
#         correct += 1

# q6 = 100 * correct / len(train)
# q6

 If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  
*


In [61]:
# Ground truth C
# Prediction C A B
# q7 = 1.0
# q7


If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  

In [62]:
# Ground truth B
# Prediction D B E
# q8 = 1/2
# q8


The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [63]:
# freq_order = freq.index.tolist()

# top3 = freq_order[:3]

# def apk(actual, pred):
#     for i,p in enumerate(pred[:3]):
#         if p == actual:
#             return 1/(i+1)
#     return 0

# scores = [
#     apk(ans, top3)
#     for ans in train["answer"]
# ]

# q9 = np.mean(scores)
# q9

The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [64]:
# scores = []

# for _, row in train.iterrows():

#     p = vectorizer.transform([row["prompt"]])

#     sims = []

#     for opt in ["A","B","C","D","E"]:
#         o = vectorizer.transform([row[opt]])
#         sims.append(
#             (opt, cosine_similarity(p,o)[0][0])
#         )

#     sims.sort(key=lambda x:x[1], reverse=True)

#     pred = [x[0] for x in sims[:3]]

#     scores.append(
#         apk(row["answer"], pred)
#     )

# q10 = np.mean(scores)
# q10

* # Milestone 2

In [65]:
# from datasets import load_dataset

# dataset = load_dataset(
#     "csv",
#     data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
# )

# train = dataset["train"]

In [66]:
# def combine(example):
#     example["combined_text"] = example["prompt"] + " " + example["A"]
#     return example

# train = train.map(combine)

# print(len(train[51]["combined_text"]))

In [67]:
# from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# print(tokenizer.vocab_size)

In [68]:
# print(tokenizer.sep_token)
# print(tokenizer.sep_token_id)

In [69]:
# prompts = train["prompt"][:]

# # or
# # prompts = list(train["prompt"])

# encodings = tokenizer(
#     prompts,
#     padding="max_length",
#     truncation=True,
#     max_length=128,
#     return_tensors="pt"
# )

# print(encodings["input_ids"].shape)

In [70]:
# from transformers import AutoModel

# model = AutoModel.from_pretrained("bert-base-uncased")

# inputs = tokenizer(train[0]["prompt"], return_tensors="pt")

# outputs = model(**inputs)

# print(outputs.last_hidden_state.shape)

In [71]:
# cls = outputs.last_hidden_state[0,0]

# print(round(cls[:5].sum().item(),4))

In [72]:
# model = AutoModel.from_pretrained(
#     "bert-base-uncased",
#     output_attentions=True
# )

# text = "Light-ion fusion is a technique."

# inputs = tokenizer(text, return_tensors="pt")

# tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

# print(tokens)

# outputs = model(**inputs)

# attention = outputs.attentions[-1][0,0]

# fusion_index = tokens.index("fusion")

# print(round(attention[0,fusion_index].item(),4))

In [73]:
# from sentence_transformers import SentenceTransformer, util

# model = SentenceTransformer(
#     "sentence-transformers/all-MiniLM-L6-v2"
# )

# prompt = train[0]["prompt"]
# option = train[0]["B"]

# embeddings = model.encode(
#     [prompt, option],
#     convert_to_tensor=True
# )

# score = util.cos_sim(
#     embeddings[0],
#     embeddings[1]
# )

# print(round(score.item(),4))

In [74]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity

In [75]:
# labels = ["A","B","C","D","E"]
# def map3_score(predictions, truths):
#     total = 0

#     for pred, truth in zip(predictions, truths):
#         if truth in pred:
#             rank = pred.index(truth) + 1
#             total += 1/rank

#     return total/len(truths)

In [76]:
# tfidf_predictions = []

# for row in train:

#     texts = [
#         row["prompt"],
#         row["A"],
#         row["B"],
#         row["C"],
#         row["D"],
#         row["E"]
#     ]

#     vectorizer = TfidfVectorizer()

#     X = vectorizer.fit_transform(texts)

#     sims = cosine_similarity(X[0], X[1:])[0]

#     ranking = np.argsort(sims)[::-1]

#     top3 = [labels[i] for i in ranking[:3]]

#     tfidf_predictions.append(top3)

In [77]:
# minilm_predictions = []

# for row in train:

#     texts = [
#         row["prompt"],
#         row["A"],
#         row["B"],
#         row["C"],
#         row["D"],
#         row["E"]
#     ]

#     emb = model.encode(
#         texts,
#         convert_to_tensor=True
#     )

#     sims = util.cos_sim(
#         emb[0],
#         emb[1:]
#     )[0].cpu().numpy()

#     ranking = np.argsort(sims)[::-1]

#     top3 = [labels[i] for i in ranking[:3]]

#     minilm_predictions.append(top3)

In [78]:
# answers = train["answer"]

# tfidf_map3 = map3_score(
#     tfidf_predictions,
#     answers
# )

# minilm_map3 = map3_score(
#     minilm_predictions,
#     answers
# )

# print("TFIDF MAP@3:", round(tfidf_map3,4))
# print("MiniLM MAP@3:", round(minilm_map3,4))

In [79]:
# count = 0

# for tfidf, mini, ans in zip(
#     tfidf_predictions,
#     minilm_predictions,
#     answers
# ):

#     if ans not in tfidf and ans in mini:
#         count += 1

# print("Count:", count)

In [80]:
# from transformers import pipeline

# classifier = pipeline(
#     "zero-shot-classification"
# )

# result = classifier(
#     train[1]["prompt"],
#     candidate_labels=[
#         train[1]["A"],
#         train[1]["B"],
#         train[1]["C"]
#     ]
# )

# print(result)

In [81]:
# result2 = classifier(
#     train[1]["prompt"],
#     candidate_labels=[
#         train[1]["A"],
#         train[1]["B"],
#         train[1]["C"]
#     ],
#     multi_label=True
# )

# softmax_sum = sum(result["scores"])
# sigmoid_sum = sum(result2["scores"])

# print(abs(softmax_sum-sigmoid_sum))

In [82]:
# from transformers import pipeline

# generator = pipeline(
#     "text-generation",
#     model="google/flan-t5-small",
#     tokenizer="google/flan-t5-small"
# )

In [83]:
# prompt = f"""Question: {train[0]['prompt']}. Is the correct answer A: {train[0]['A']} or B: {train[0]['B']}? Answer with just the letter A or B."""

# result = generator(
#     prompt,
#     max_new_tokens=5
# )

# print(result)

In [84]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
# model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

In [85]:
# prompt = f"""Question: {train[0]['prompt']}. Is the correct answer A: {train[0]['A']} or B: {train[0]['B']}? Answer with just the letter A or B."""

# inputs = tokenizer(prompt, return_tensors="pt")

# outputs = model.generate(
#     **inputs,
#     max_new_tokens=5
# )

# answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

# print(answer)

* # Milestone 3

In [86]:
# !pip install faiss-cpu #install FAISS

# import pandas as pd 
# import numpy as np 
# import faiss 
# from sentence_transformers import SentenceTransformer, CrossEncoder 
# from transformers import AutoTokenizer, pipeline 
# from sklearn.feature_extraction.text import TfidfVectorizer 
# from sklearn.metrics.pairwise import cosine_similarity 

# train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

# print("Creating nowledge base")
# kb = [] 
# for idx, row in train.iterrows(): 
#     correct_letter = row['answer'] 
#     kb.append(str(row[correct_letter])) 

# print("Loading embedding model and creating index") 
# model = SentenceTransformer('all-MiniLM-L6-v2') 
# kb_embeddings = model.encode(kb, show_progress_bar=False) 
# index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
# index.add(kb_embeddings)

# print("Knowledge base successfully created")

In [87]:
# zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
# row_150 = train.iloc[150] 
# prompt_150 = str(row_150['prompt']) 
# labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
# ans_150 = str(row_150[row_150['answer']])

In [88]:
# result = zs(prompt_150, candidate_labels=labels_150)

# correct_option = ans_150
# score = result["scores"][result["labels"].index(correct_option)]

# print(round(score, 3))

In [89]:
# query_embedding = model.encode([prompt_150])

# D, I = index.search(query_embedding, 10)

# retrieved_indices = I[0]

# true_doc = kb[150]

# for rank, idx in enumerate(retrieved_indices, start=1):
#     if kb[idx] == true_doc:
#         print(rank)

In [90]:
# cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# docs_10 = [kb[i] for i in retrieved_indices]

# pairs = [[prompt_150, doc] for doc in docs_10]

# scores = cross_encoder.predict(pairs)

# ranking = np.argsort(scores)[::-1]

# true_doc = kb[150]

# for rank, idx in enumerate(ranking, start=1):
#     if docs_10[idx] == true_doc:
#         print(rank)

In [91]:
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# row = train.iloc[42]

# prompt = str(row["prompt"])

# query_embedding = model.encode([prompt])

# _, I = index.search(query_embedding, 5)

# docs = [kb[i] for i in I[0]]

# context = " ".join(docs)

# rag = f"Context: {context} Question: {prompt}"

# tokens = tokenizer(rag, truncation=False)

# print(len(tokens["input_ids"]))

In [92]:
# true_doc = kb[150]

# rag = f"Context: {true_doc} Question: {prompt_150}"

# result = zs(rag, candidate_labels=labels_150)

# score = result["scores"][result["labels"].index(ans_150)]

# print(round(score,3))

In [93]:
# wrong_doc = kb[999]

# rag = f"Context: {wrong_doc} Question: {prompt_150}"

# result = zs(rag, candidate_labels=labels_150)

# score = result["scores"][result["labels"].index(ans_150)]

# print(round(score,3))

In [94]:
# hits = 0

# for i in range(100):

#     row = train.iloc[i]

#     prompt = str(row["prompt"])

#     correct_doc = str(row[row["answer"]])

#     query_embedding = model.encode([prompt])

#     _, I = index.search(query_embedding, 5)

#     docs = [kb[idx] for idx in I[0]]

#     if any(correct_doc in doc for doc in docs):
#         hits += 1

# hit_rate = hits / 100 * 100

# print(round(hit_rate,1))

In [95]:
# cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# scores_all = []

# for i in range(20):

#     row = train.iloc[i]

#     prompt = str(row["prompt"])

#     labels = [
#         str(row["A"]),
#         str(row["B"]),
#         str(row["C"]),
#         str(row["D"]),
#         str(row["E"])
#     ]

#     correct_letter = row["answer"]

#     embedding = model.encode([prompt])

#     _, I = index.search(embedding, 5)

#     docs = [kb[idx] for idx in I[0]]

#     pairs = [[prompt, d] for d in docs]

#     ce_scores = cross_encoder.predict(pairs)

#     best_doc = docs[np.argmax(ce_scores)]

#     rag = f"Context: {best_doc} Question: {prompt}"

#     result = zs(rag, candidate_labels=labels)

#     ranked = result["labels"]

#     letter_map = {
#         str(row["A"]): "A",
#         str(row["B"]): "B",
#         str(row["C"]): "C",
#         str(row["D"]): "D",
#         str(row["E"]): "E"
#     }

#     predicted_letters = [letter_map[x] for x in ranked[:3]]

#     if correct_letter == predicted_letters[0]:
#         ap = 1
#     elif correct_letter == predicted_letters[1]:
#         ap = 1/2
#     elif correct_letter == predicted_letters[2]:
#         ap = 1/3
#     else:
#         ap = 0

#     scores_all.append(ap)

# print(round(np.mean(scores_all),3))

* # Milestone 4 

In [96]:
# label_map = {"A":0,"B":1,"C":2,"D":3,"E":4}
# label_map[train.loc[150, "answer"]]   # -> 2

In [97]:
# len(str(train.loc[0,"prompt"]) + " [SEP] " + str(train.loc[0,"B"]))   # -> 407

In [98]:
# input_ids.shape
# torch.Size([1, 5, 128])
# Second dimension = 5

In [99]:
# 16 * 5 * 128   # -> 10240

In [100]:
# outputs.logits.shape
# torch.Size([1, 5])
# 5 logits

In [101]:
# outputs.loss.dim()   # -> 0

In [102]:
# sum(p.numel() for p in model.parameters() if p.requires_grad)
# # -> 294912

In [103]:
# dataset[0]["input_ids"].shape
# # (5, 128)
# # Number of choices = 5

In [104]:
# with torch.no_grad():
#     outputs = model(
#         input_ids=input_ids,
#         attention_mask=attention_mask
#     )

# probs = torch.softmax(outputs.logits, dim=1)
# round(probs[0, 4].item(), 4)